# Sweden deals: reconciliation and what we need from Intrum

One row per Swedish deal we own. Heimdall deals use Intrum's id and Lokki deals use ours, so
`LOK-SE25008` is the same book Intrum calls `SE25018`.

For each deal you get the case count, whether we have a seller tape and how many cases it matches,
and whether CAT reconciles to PAM, both today and after we fix our two dbt bugs.

A deal reconciles when CAT has payments in at least 80% of the months PAM does and the total is
within 10% of PAM, counting from the acquisition date. It's the same test the Data Landscape
workbook uses.

| What | Where it comes from | Run time |
|---|---|---|
| Deals, cases, PAM, CAT today | `queries/sweden_deals.sql` | about a minute |
| CAT with both bugs fixed | `queries/sweden_raw_cat.sql`, saved in `.cache/` | about 20 minutes, so it only reruns if you set `REFRESH_RAW = True` |
| Seller tapes | `seller_tapes.py`, reading the asset-index folders (headers only) | about a minute |
| Tape match counts | typed in below, measured 16 Nov 2026 | |


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

sys.path.insert(0, "../../packages")
import datalake
import seller_tapes

REFRESH_RAW = False  # True reruns the 20-minute raw CAT query instead of reading .cache/
CONNECTION, WAREHOUSE = "dev", "DEV_WH"  # PROD_RAW is only readable from the sandbox
CASH_TOLERANCE, MIN_MONTHS = 0.10, 0.80
pd.set_option("display.float_format", lambda v: f"{v:,.5f}")
pd.set_option("display.max_columns", None)


## 1. Deals, cases, PAM and CAT today

In [ ]:
deals = datalake.q("work/hackathon/queries/sweden_deals.sql",
                   connection=CONNECTION, warehouse=WAREHOUSE, lower=True)
deals["our_id"] = deals["deal_key"].str[4:]
print(f"{len(deals):,} Swedish deals, queried {pd.Timestamp.now():%Y-%m-%d %H:%M}")


## 2. CAT with our two bugs fixed

Bug 1: we drop payments where `booking_ind = 'Y'`, and on system 1062 that is every payment.
Bug 2: we de-duplicate on `orig_tr_id`, and on system 2080 one payment can come in under several
ids, so it gets counted more than once. This rebuilds CAT from the raw feed without either
mistake. Which bug hits a deal depends on which system its cases are on.

In [ ]:
cache = Path(".cache/sweden_raw_cat.csv")
if REFRESH_RAW or not cache.exists():
    fresh = datalake.q("work/hackathon/queries/sweden_raw_cat.sql",
                       connection=CONNECTION, warehouse=WAREHOUSE, lower=True)
    cache.parent.mkdir(exist_ok=True)
    fresh.to_csv(cache, index=False)
print(f"raw CAT: {'queried' if REFRESH_RAW else 'from cache'}, "
      f"{pd.Timestamp.fromtimestamp(cache.stat().st_mtime):%Y-%m-%d %H:%M}")
raw = pd.read_csv(cache)[[
    "deal_key", "cat_cases", "cases_on_1062", "cases_on_2080",
    "fixed_collections", "fixed_months",
]].rename(columns={"cat_cases": "raw_cat_cases"})


## 3. Seller tapes

Looks in the asset-index inbox and sorted folders, and in `work/workshop/asset-index/`.
In the table, `dated payments` means the tape has every payment with its date, `totals only`
means it just has amounts paid so far, and `none` means no payment information at all. Only
dated payments can build a collection curve.

In [ ]:
tapes = seller_tapes.per_deal(deals, ["our_id", "intrum_id"])
display(tapes)


## 4. How many cases each tape matches

Measured on 16 Nov 2026. The percentage is out of the deal's CAT cases on both Swedish systems, but each tape was only matched against one system, so the real figure may be higher. Every tape gets a match %, with or without payment history.
Rerun `just deal-bridge` after a tape is loaded again and update these.

In [ ]:
# deal_key: (matched cases, how the tape joins), 16 Nov 2026. Each was matched on one Intrum
# system only, so the other system's cases were never checked and may join another way.
TAPE_BRIDGE = {
    "HEI-SE17039": (1_572, "Aktnr = case_id, system 1062 only"),
    "HEI-SE19026": (8_312, "tape id = invoice_number, system 2080 only"),
    "HEI-SE20009": (0, "no id joins; balances matched about 88% in an earlier check"),
    "LOK-SE25008": (6_267, "Case ID (client), via Intrum's mapping file; batch SE25018D010000 missing"),
    "LOK-SE25013": (1_582, "invoice_number = LoanNumber, not accepted yet"),
    "LOK-SE25014": (64, "Intrum's case-mapping file, not the tape"),
}
TAPE_MATCHED_CASES = {k: v[0] for k, v in TAPE_BRIDGE.items()}


## 5. The table

In [ ]:
t = (deals.merge(raw, on="deal_key", how="left")
          .merge(tapes, on="deal_key", how="left"))
t["has_seller_tape"] = t["tape_files"].notna()
t["payment_history"] = (t["payment_history"].fillna("none")
                        .map({"dated": "dated payments", "summary": "totals only", "none": "none"}))
t["tape_with_payments"] = t["payment_history"].eq("dated payments")
# Every tape gets a match %, whether or not it has payment history: they're separate questions.
t["bridge_pct"] = 100 * t["deal_key"].map(TAPE_MATCHED_CASES) / t["raw_cat_cases"]
t["bridge_via"] = t["deal_key"].map({k: v[1] for k, v in TAPE_BRIDGE.items()})

t["reconciliation_pct"] = 100 * t["cat_collections"] / t["pam_collections"]
t["reconciliation_pct_after_fixes"] = 100 * t["fixed_collections"] / t["pam_collections"]
t["months_pct"] = 100 * t["cat_months"] / t["pam_months"]
t["months_pct_after_fixes"] = 100 * t["fixed_months"] / t["pam_months"]


def passes(cash_pct, months_pct):
    return ((cash_pct / 100 - 1).abs().le(CASH_TOLERANCE)
            & (months_pct / 100).ge(MIN_MONTHS)).fillna(False)


t["reconciles"] = passes(t["reconciliation_pct"], t["months_pct"])
t["reconciles_after_fixes"] = passes(t["reconciliation_pct_after_fixes"],
                                     t["months_pct_after_fixes"])
# The bugs only matter for a deal if today's total differs from the fixed one.
t["bugs_bite"] = ((t["cat_collections"] / t["fixed_collections"] - 1).abs() > 0.01).fillna(False)
t["which_bug"] = t.apply(
    lambda r: "bugs 1 and 2" if r.cases_on_1062 > 0 and r.cases_on_2080 > 0
    else "bug 1" if r.cases_on_1062 > 0
    else "bug 2" if r.cases_on_2080 > 0 else None, axis=1)

CANCEL_OUT = "Only passes today because our bugs cancel out, fails once fixed"


def fail_reason(r):
    if pd.isna(r.pam_collections):
        return "No PAM collections after acquisition"
    if pd.isna(r.raw_cat_cases):
        return "No cases in CAT, probably closed before Oct 2014"
    if r.reconciles and not r.reconciles_after_fixes:
        return CANCEL_OUT
    if r.reconciles:
        return "Reconciles, our bugs move the total a little" if r.bugs_bite else ""
    if r.reconciles_after_fixes:
        return f"Our {r.which_bug}, reconciles once fixed"
    # Past this point it still fails after the fixes, so it isn't our bugs.
    if abs(r.reconciliation_pct_after_fixes / 100 - 1) <= CASH_TOLERANCE:
        return "Total matches, but CAT has payments in under 80% of PAM's months"
    if r.our_id in ("SE15044", "SE16025"):
        return "CAT has more 2016-2019 cash than PAM"
    if r.acquisition_date < pd.Timestamp("2014-10-01"):
        return "Probably bought before CAT's history starts (Oct 2014)"
    if r.acquisition_date < pd.Timestamp("2020-01-01"):
        return "Probably PAM's pre-2020 reporting"
    return "Not explained yet"


t["acquisition_date"] = pd.to_datetime(t["acquisition_date"])
t["fail_reason"] = t.apply(fail_reason, axis=1)

COLUMNS = [
    "intrum_id", "dim_case_cases", "has_seller_tape", "tape_with_payments", "payment_history",
    "bridge_pct", "bridge_via", "cat_collections", "pam_collections", "reconciles", "reconciliation_pct",
    "fail_reason", "reconciles_after_fixes", "reconciliation_pct_after_fixes",
    # supporting
    "months_pct", "months_pct_after_fixes", "fixed_collections", "raw_cat_cases",
    "which_bug", "bugs_bite", "acquisition_date", "tape_found_in", "unreadable_files",
]
table = (t.set_index("deal_key")[COLUMNS]
          .sort_values(["tape_with_payments", "has_seller_tape", "dim_case_cases"],
                       ascending=False))
table.to_csv("sweden_status.csv")
display(table)


## 6. Totals

In [ ]:
summary = pd.Series({
    "Swedish deals we own": len(table),
    "with a seller tape": int(table["has_seller_tape"].sum()),
    "with a tape that has payment history": int(table["tape_with_payments"].sum()),
    "reconcile today": int(table["reconciles"].sum()),
    "reconcile after our fixes": int(table["reconciles_after_fixes"].sum()),
    "pass today but fail once fixed": int(table["fail_reason"].eq(CANCEL_OUT).sum()),
    "still fail after our fixes": int((~table["reconciles_after_fixes"]).sum()),
})
display(summary.to_frame("deals"))
display(table.groupby("fail_reason", dropna=False)
             .agg(deals=("intrum_id", "size"), cases=("dim_case_cases", "sum"))
             .sort_values("cases", ascending=False))


## 7. Questions for Intrum

Intrum sends CAT, PAM and the case-mapping files, and has the seller tapes from its own purchases,
so most of what's left is a question for them. The deal lists and counts come from the table above
and update when it reruns.

Bugs 1 and 2 are ours. Intrum's data is fine there, we just read two fields wrongly, so the first
two questions only check what those fields mean. Even with both bugs fixed, 524 of the 639 deals
still don't reconcile, and questions 3 to 9 are about those.

In [ ]:
def deals_with(mask, n=None):
    rows = table[mask].sort_values("dim_case_cases", ascending=False)
    ids = list(rows.index if n is None else rows.index[:n])
    more = len(rows) - len(ids)
    return ", ".join(ids) + (f" (+{more} more)" if more > 0 else ""), len(rows), rows["dim_case_cases"].sum()


reason = table["fail_reason"].fillna("")
asks = []


def ask(theme, question, why, mask=None, deals=None, n=12):
    found = deals_with(mask, n) if mask is not None else (deals, None, None)
    asks.append({"theme": theme, "ask": question, "why": why, "deals": found[0],
                 "n_deals": found[1], "cases": found[2]})


ask("Fields",
    "What does booking_ind mean? On system 1062 every payment row is 'Y', on 2080 it's always "
    "empty. Should 'Y' rows count as collections?",
    "We drop 'Y' rows, which removes all of system 1062's cash (bug 1).",
    mask=table["which_bug"].isin(["bug 1", "bugs 1 and 2"]) & table["bugs_bite"])
ask("Fields",
    "What identifies a single payment? On 2080 the same payment can come in under several "
    "orig_tr_id values (case 2080-1212023 has 14 rows of 570.00 on 2016-01-27). How do we tell a "
    "resend or a correction from a new payment?",
    "We de-duplicate on orig_tr_id, so some 2080 payments are counted more than once (bug 2).",
    mask=table["which_bug"].isin(["bug 2", "bugs 1 and 2"]) & table["bugs_bite"])
ask("Fields",
    "35% of payment rows have no transaction_date. Which date should we use for them?",
    "Without a date a payment can't go on a collection curve or be compared with PAM.",
    deals="all of CAT")

ask("Reconciliation",
    "These deals were all bought between 2001 and 2010. PAM has collections for them but CAT has "
    "no cases. Were they closed before the move to CAT in October 2014, or are they serviced "
    "somewhere else?",
    "No cases means we can't model them at loan level. The Oct 2014 explanation is our guess "
    "from the dates.",
    mask=table["raw_cat_cases"].isna())
ask("Reconciliation",
    "Can you send CAT payment history from before the October 2014 migration?",
    "Books bought before then come out low against PAM. Our guess from the dates, not checked "
    "deal by deal.",
    mask=reason.str.contains("Oct 2014") & table["raw_cat_cases"].notna())
ask("Reconciliation",
    "What does PAM's COLLTOTAL leave out before 2020? On SE15044 and SE16025 CAT has more "
    "2016-2019 cash than PAM, and from 2020 they match.",
    "Checked year by year on those two books. The others here are a guess from the dates.",
    mask=reason.str.contains("2016-2019|pre-2020"))
ask("Reconciliation",
    "For these deals, which months does PAM show collections that CAT has no payments for?",
    "The totals match but CAT is missing months.",
    mask=reason.str.contains("under 80%"))
ask("Reconciliation",
    "CAT and PAM differ on these deals and our bugs don't explain it. Can you help us find why?",
    "Includes LOK-SE25013 (Norion), which has a tape, and the deals that only pass today because "
    "our two bugs cancel out.",
    mask=reason.isin(["Not explained yet", CANCEL_OUT]))
ask("Reconciliation",
    "Why does CAT have 152% of the principal PAM says we bought on SE25018?",
    "Its collections only match PAM because of it.", deals="LOK-SE25008 (Intrum SE25018)")

ask("Mappings",
    "Please add batch SE25018D010000 (3,446 cases) to the case-mapping file "
    "(aix_cm_overview_brocc_311026.xlsx). The overview says it's mapped but there are no rows.",
    "That takes the SE25008 tape from 45% of cases to the whole deal.", deals="LOK-SE25008")
ask("Mappings",
    "Can you send the CAT case ids for the Project Turing tape? Every reference column CAT has "
    "for this book is empty.",
    "The tape has payment history and the deal reconciles, but nothing joins.",
    deals="HEI-SE20009")
ask("Mappings",
    "Is CAT invoices.invoice_number the same as LoanNumber in the Norion tape? 1,582 of 1,583 "
    "match.",
    "If yes, we can use it to join the tape.", deals="LOK-SE25013")
ask("Mappings",
    "Which of client_ref_1, client_ref_2 and client_number holds the seller's account number?",
    "The seller's id is in one of them, but it isn't the same one on every book.",
    deals="all Swedish")

ask("Code lists",
    "The full, current list of case status codes with descriptions, and which ones mean open, "
    "closed, legal, enforcement and insolvency. Is there anything newer than the 112-code "
    "document we have?",
    "22.4m cases have a status code and we can't decode any of them. We have a draft mapping "
    "that you haven't confirmed.", deals="all Intrum")
ask("Code lists",
    "The list of closing codes, with descriptions and what each one means (paid in full, "
    "settled, written off, returned to client, insolvency, deceased and so on).",
    "64.5% of current Swedish cases (1,227,804 of 1,904,473) have one of 15 closing codes, and "
    "we can't decode them.", deals="all Swedish")
ask("Code lists",
    "The list of phase codes with descriptions, and phase filled in for Swedish cases. On "
    "system 2080 most of them are NA.",
    "Phase is what tells us amicable, legal or enforcement. We only know the A / L / E prefix, "
    "from a conversation, not a document.", deals="all Swedish")
ask("Code lists",
    "The legal step and legal result code lists, and which steps mean a court filing, a "
    "judgment or a Kronofogden case.",
    "41.2m legal steps are sitting unused because we can't read the codes. They're how we'd "
    "know a case is in court.", deals="all Intrum")

ask("Seller tapes",
    "Copies of the Norion OffShore files without the DRM protection. We can't open them.",
    "They have the dated cash flow for the book.", deals="LOK-SE25013")
ask("Seller tapes",
    "The dated payment history behind these tapes. What we have only shows totals.",
    "We can't build a collection curve from a total.",
    mask=table["payment_history"].eq("totals only"))
ask("Seller tapes",
    "The seller tapes from your own purchase of these books, with payment history, largest "
    "first.",
    "These reconcile once our bugs are fixed, so with a tape we could model them end to end.",
    mask=table["reconciles_after_fixes"] & ~table["has_seller_tape"], n=15)
ask("Seller tapes",
    "Is there any seller tape for SE15044 or SE16025?",
    "Two of the biggest Swedish books, and we can't find a tape for either.",
    deals="HEI-SE15044, HEI-SE16025")

intrum_asks = pd.DataFrame(asks)
intrum_asks.index = range(1, len(intrum_asks) + 1)
intrum_asks.to_csv("intrum_asks.csv", index_label="#")
with pd.option_context("display.max_colwidth", None):
    display(intrum_asks)


## How far to trust this

The deal list, case counts, PAM, CAT today and CAT after the fixes are all queried directly. The
fixed CAT figure uses the same rebuild that matched PAM to 99.88% on the Portuguese books.

The tape match counts are from 16 Nov 2026, and the raw CAT figures are from whenever the cache
was last refreshed (the date prints in section 2).

Anything starting with "Probably" is a guess from the acquisition date. Only SE15044 and SE16025
have been checked year by year.

Which bug applies is worked out from the system a deal's cases are on: bug 1 only happens on
1062 and bug 2 only on 2080. That was checked on six books in
`../data-issues/intrum/queries/dbt_vs_ours_vs_pam.sql`.
